# LFM2 vocabulary expansion — T4 smoke test

This follows the LFM2.5 recipe at small scale: continue BPE on Hindi text, initialize new rows from old sub-token rows, and train only the new rows. It intentionally runs only Stage 1; the paper's full Stage 2 requires continued pretraining at much larger scale.

In [ ]:
!pip -q install transformers datasets accelerate peft sentencepiece
!git clone -q https://github.com/Lexsi-Labs/aligntune.git /content/aligntune
import sys
sys.path.insert(0, '/content/aligntune/src')
sys.path.insert(0, '/content/aligntune/third_party/tokenizer-extension')

In [ ]:
import copy
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import TrainableTokensConfig, get_peft_model
from aligntune.core.tokenization.vocab.continued_bpe import extend_tokenizer_continued_bpe
from aligntune.core.tokenization.embedding_adaptation import adapt_token_embeddings
def resolve_new_token_ids(old_tokenizer, new_tokenizer):
    old_vocab = old_tokenizer.get_vocab()
    return sorted(token_id for token, token_id in new_tokenizer.get_vocab().items()
                  if token not in old_vocab)

def resolve_input_embedding_module_name(model):
    input_embedding = model.get_input_embeddings()
    for name, module in model.named_modules():
        if module is input_embedding:
            return name
    raise ValueError('input embedding module not found')

MODEL_ID = 'LiquidAI/LFM2-350M'
LANG_CONFIG = '20231101.hi'
MAX_CORPUS_SAMPLES = 256
NEW_TOKENS = 512
MAX_LENGTH = 256
TRAIN_STEPS = 100
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
print(torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')

In [ ]:
# Use a small Hindi Wikipedia slice for a fast, real-language smoke test.
stream = load_dataset('wikimedia/wikipedia', LANG_CONFIG, split='train', streaming=True)
corpus = [row['text'] for row in stream.take(MAX_CORPUS_SAMPLES) if row.get('text')]
print('documents:', len(corpus), 'characters:', sum(map(len, corpus)))

In [ ]:
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
new_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
old_vocab_size = len(base_tokenizer)

extension = extend_tokenizer_continued_bpe(
    new_tokenizer, corpus, num_new_tokens=NEW_TOKENS,
    is_sentencepiece=False, show_progress=True,
)
new_token_ids = resolve_new_token_ids(base_tokenizer, new_tokenizer)
print({
    'old_vocab': old_vocab_size,
    'new_vocab': len(new_tokenizer),
    'new_ids': len(new_token_ids),
    'unreachable': extension['unreachable_tokens'],
})
assert new_token_ids and max(new_token_ids) < len(new_tokenizer)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map='auto' if DEVICE == 'cuda' else None,
)
if DEVICE == 'cpu':
    model.to(DEVICE)

# FVT: copy retained rows and mean-initialize only added rows.
adapt_token_embeddings(
    model, base_tokenizer, new_tokenizer,
    method='mean_of_constituents', pad_to_multiple_of=128,
)
embedding_rows = model.get_input_embeddings().weight.shape[0]
print({'tokenizer_rows': len(new_tokenizer), 'embedding_rows': embedding_rows})
assert embedding_rows >= len(new_tokenizer)

In [ ]:
# Native PEFT TrainableTokens: this does NOT add LoRA.
config = TrainableTokensConfig(
    token_indices=new_token_ids,
    target_modules=resolve_input_embedding_module_name(model),
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, config)
trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
print('trainable parameters:', [(n, tuple(p.shape)) for n, p in trainable])
assert trainable and all('trainable_tokens_delta' in n for n, _ in trainable)
assert all('lora_' not in n for n, _ in trainable)

In [ ]:
# Tokenize the adaptation corpus and train only the selected new-token deltas.
texts = corpus[: min(len(corpus), 128)]
batch = new_tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LENGTH)
input_ids = batch['input_ids'].to(model.device)
attention_mask = batch['attention_mask'].to(model.device)
optimizer = torch.optim.AdamW([p for _, p in trainable], lr=2e-3, weight_decay=0.0)
model.train()
for step in range(TRAIN_STEPS):
    optimizer.zero_grad(set_to_none=True)
    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
    out.loss.backward()
    optimizer.step()
    if step % 20 == 0 or step == TRAIN_STEPS - 1:
        print(f'step={step:03d} loss={out.loss.item():.4f}')

In [ ]:
def fertility(tokenizer, docs):
    words = sum(len(x.split()) for x in docs)
    tokens = sum(len(tokenizer.encode(x, add_special_tokens=False)) for x in docs)
    return tokens / max(words, 1)

print({
    'fertility_before': fertility(base_tokenizer, texts),
    'fertility_after': fertility(new_tokenizer, texts),
    'new_token_count': len(new_token_ids),
    'embedding_rows_including_padding': model.get_input_embeddings().weight.shape[0],
})
print('Stage 1 smoke test complete. Stage 2 would unfreeze the full model for continued pretraining.')